In [2]:
import tkinter as tk
import random

# Game settings
WIDTH = 400
HEIGHT = 400
BLOCK = 20

actions = ["UP", "DOWN", "LEFT", "RIGHT"]

# RL parameters
alpha = 0.1
gamma = 0.9
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.01

q_table = {}

def get_state(snake, food):
    head = snake[0]
    return (head[0] - food[0], head[1] - food[1])

def choose_action(state):
    if random.random() < epsilon:
        return random.choice(actions)
    if state not in q_table:
        q_table[state] = {a:0 for a in actions}
    return max(q_table[state], key=q_table[state].get)

def update_q(state, action, reward, next_state):
    if state not in q_table:
        q_table[state] = {a:0 for a in actions}
    if next_state not in q_table:
        q_table[next_state] = {a:0 for a in actions}

    old = q_table[state][action]
    next_max = max(q_table[next_state].values())

    q_table[state][action] = old + alpha * (reward + gamma * next_max - old)

def reset():
    snake = [[100,100]]
    food = [random.randrange(0, WIDTH, BLOCK), random.randrange(0, HEIGHT, BLOCK)]
    return snake, food

# Tkinter setup
root = tk.Tk()
root.title("RL Snake (Tkinter)")
canvas = tk.Canvas(root, width=WIDTH, height=HEIGHT, bg="white")
canvas.pack()

snake, food = reset()

def draw():
    canvas.delete("all")
    for s in snake:
        canvas.create_rectangle(s[0], s[1], s[0]+BLOCK, s[1]+BLOCK, fill="green")
    canvas.create_rectangle(food[0], food[1], food[0]+BLOCK, food[1]+BLOCK, fill="red")

def game_step():
    global snake, food, epsilon

    state = get_state(snake, food)
    action = choose_action(state)

    head = snake[0].copy()

    if action == "UP":
        head[1] -= BLOCK
    elif action == "DOWN":
        head[1] += BLOCK
    elif action == "LEFT":
        head[0] -= BLOCK
    elif action == "RIGHT":
        head[0] += BLOCK

    reward = -1

    if head == food:
        reward = 10
        snake.insert(0, head)
        food = [random.randrange(0, WIDTH, BLOCK), random.randrange(0, HEIGHT, BLOCK)]
    else:
        snake.insert(0, head)
        snake.pop()

    # Wall collision
    if head[0] < 0 or head[0] >= WIDTH or head[1] < 0 or head[1] >= HEIGHT:
        reward = -100
        snake, food = reset()

    next_state = get_state(snake, food)
    update_q(state, action, reward, next_state)

    # Reduce exploration
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    draw()
    root.after(100, game_step)

game_step()
root.mainloop()